# Ping / RTT Exploratory Analysis

Demonstrates loading ping data and running exploratory analysis to identify host performance, detect anomalies, and reveal diurnal patterns.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib
matplotlib.use('Agg')
%matplotlib inline

from inventor_analysis.loaders import load_ping_data
from inventor_analysis.ping_analysis import (
    compute_host_statistics,
    compute_daily_trends,
    compute_hourly_patterns,
    detect_anomalies_zscore,
    rank_hosts,
    compute_reliability_scores,
)
from inventor_analysis.visualization import (
    plot_rtt_distribution,
    plot_host_comparison,
    plot_daily_trend,
    plot_hourly_pattern,
    plot_anomaly_timeline,
)

## Load Data

Point at a directory containing Inventor ping JSONL files. Uses `sample_data/` by default — replace with a full data directory for production analysis.

In [ ]:
df = load_ping_data("../sample_data", max_files=5)

print(f"Loaded {len(df):,} records, {df['target_host'].nunique()} hosts")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## Host Performance Statistics

Per-host aggregation: mean, std, median, p95, p99 for RTT, plus jitter and packet loss.

In [ ]:
stats = compute_host_statistics(df)
stats

## Host Rankings

In [ ]:
fastest = rank_hosts(df, metric='rtt_avg', top_n=5, ascending=True)
slowest = rank_hosts(df, metric='rtt_avg', top_n=5, ascending=False)

print(f"Fastest 5 hosts: {fastest}")
print(f"Slowest 5 hosts: {slowest}")

## Daily Trends

In [ ]:
daily = compute_daily_trends(df)
daily

## Hourly Patterns (Diurnal Cycles)

In [ ]:
hourly = compute_hourly_patterns(df)
hourly

## Anomaly Detection (Z-Score)

In [ ]:
df_anomalies = detect_anomalies_zscore(df, threshold=3.0)
anomalies = df_anomalies[df_anomalies['is_anomaly']]

print(f"Total anomalies: {len(anomalies):,} ({len(anomalies)/len(df)*100:.2f}%)")

if not anomalies.empty:
    print("\nTop anomalies by RTT:")
    display(anomalies.nlargest(5, 'rtt_avg')[['timestamp', 'target_host', 'rtt_avg', 'jitter', 'pkts_lost']])

## Reliability Scores

In [ ]:
reliability = compute_reliability_scores(df)
reliability

## Visualizations

In [ ]:
plot_rtt_distribution(df, source_label='Internet')

In [ ]:
plot_host_comparison(df, metric='rtt_avg', top_n=10)

In [ ]:
plot_daily_trend(daily, metric='rtt_mean', label='Internet')

In [ ]:
plot_hourly_pattern(hourly, metric='rtt_mean')